# Model Performance Analysis

Post-hoc analysis of model behavior on multiple-choice tasks. **This notebook does not run inference** — it reads JSONL files written by [`finetune/run_evaluations.py`](../finetune/run_evaluations.py) and turns them into accuracy / calibration / position-bias / confidence-distribution plots.

## How to use

1. **Generate eval data** on Runpod / vast.ai. `run_evaluations.py` is config-driven (no CLI flags) — edit `EvalConfig` in [`finetune_config.py`](../finetune_config.py) and run:
   ```bash
   python finetune/run_evaluations.py
   ```
   That writes one JSONL per dataset to `outputs/evaluations/<timestamp>_<model>_<dataset>_n<N>.jsonl` (and a paired `.txt` run record).

2. **Pull the JSONL(s) locally** and list them in `EVAL_PATHS` below. Each entry becomes one labeled run on every plot — labels are `model_type / dataset` (e.g. `instruct / TriviaMC`, `finetuned / SimpleMC`), parsed from the filename and the sample-row `type` field. Leave the list empty to fall back to the latest file in `outputs/evaluations/`.

3. **Run the cells.** Plots and tables fall out per-section, with one panel/line per file you listed.

## What's in each section

| Section | What it answers |
| --- | --- |
| 1. Load eval results | Read each JSONL, tag with model_type + dataset, summarize counts. |
| 2. Accuracy & position bias | Overall accuracy by run; per-position conditional accuracy; chi-square uniformity test on predicted positions; one bar-chart panel per run. |
| 3. Calibration & metacognition | Reliability diagram on `top_prob` (overlaid across runs); Spearman(entropy, stated_conf) per run; **three overlaid plots** for entropy ↔ stated_conf, stated_conf ↔ correctness, and entropy ↔ correctness. |
| 4. Confidence answer distribution | Bar chart of `predicted_confidence_bin_index` (self) and `predicted_other_confidence_bin_index` (other) per run — checks whether the model is collapsing onto a single confidence bin. |
| 5. Entropy distribution | Overlaid histograms of MC-answer entropy per run — shows whether models are getting peakier or flatter across conditions. |
| 6. Delegation game | **Inactive** — delegate-game scripts were dropped. Stub kept for when it's rebuilt. |
| 7. ECT analysis | Reads `outputs/ECT/*.json` from the ECT runner; reports ECE/AUROC/Brier vs. an MC-entropy ceiling. |

## Notes for migrating from `analyze_activations.ipynb`

Sections 1.8 / 1.9 / 1.10 used to live in `analyze_activations.ipynb` and read from `paired_data.json` (the introspection-experiment output). Those exact cells are preserved in [`legacy_code/analyze_activations_pre_split.ipynb`](legacy_code/analyze_activations_pre_split.ipynb). The new convention here is to read `run_evaluations.py` JSONLs instead. They expose fields the legacy `paired_data.json` didn't (e.g., `probs_ABCD`, `top_prob`, `prob_gap`, `is_correct`, `expected_confidence`), so code is a little simpler.

In [ ]:
# Papermill parameters cell. Defaults are used when running this notebook
# directly in Jupyter; papermill injects an override cell below this one
# when invoked via analysis_helpers.run_notebook() / the papermill CLI.

# List of JSONL files produced by finetune/run_evaluations.py. Each file is
# treated as one run; plots overlay/compare across runs and label each by
# (model_type, dataset) parsed from the filename + sample 'type' field.
# Leave EVAL_PATHS empty to fall back to the latest file in outputs/evaluations/.
EVAL_PATHS = [
    # 'outputs/evaluations/2026-05-03-17-49-00_meta-llama-Llama-3.1-8B-Instruct_instruct_PopMC_n14267.jsonl',
    # 'outputs/evaluations/2026-05-03-18-00-04_meta-llama-Llama-3.1-8B-Instruct_instruct_TriviaMC_n2416.jsonl',
    # 'outputs/evaluations/2026-05-03-17-59-22_meta-llama-Llama-3.1-8B-Instruct_instruct_SimpleMC_n500.jsonl',
    'data/balanced_metacognition.jsonl',
    'data/balanced_metacognition_train.jsonl',
    'data/balanced_metacognition_val.jsonl',
    'data/balanced_metacognition_test.jsonl',
]

# Free-form label that gets stamped into the notebook so you can tell
# saved-output copies apart at a glance.
RUN_LABEL = "interactive"


## Section 1 — Load eval results

In [ ]:
import json
import re
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Make analysis_helpers and core importable when this notebook lives in analysis/.
REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'analysis' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / 'analysis') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'analysis'))

from analysis_helpers import (
    load_evaluation_jsonl, latest_eval_log,
    calibration_table, expected_calibration_error, brier_score,
)


def _resolve(p):
    p = Path(p)
    return p if p.is_absolute() else REPO_ROOT / p


def _parse_meta(path):
    """Return (model_type, dataset) for an eval JSONL.

    model_type comes from the first sample row's 'type' field
    (e.g. 'instruct_eval_sample' -> 'instruct'). dataset is parsed from the
    filename which is built as
    `<timestamp>_<base_model_safe>_<model_type|adapter>_<dataset>_n<N>.jsonl`.
    """
    path = Path(path)
    model_type = "unknown"
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            t = json.loads(line).get("type", "")
            if t.endswith("_eval_sample"):
                model_type = t[: -len("_eval_sample")]
                break
    stem = path.stem
    m = re.match(r"^(.*)_n\d+$", stem)
    core = m.group(1) if m else stem
    dataset = core.rsplit("_", 1)[-1] if "_" in core else "unknown"
    return model_type, dataset


# Resolve paths (fall back to latest file if EVAL_PATHS is empty).
if not EVAL_PATHS:
    latest = latest_eval_log(REPO_ROOT / 'outputs' / 'evaluations')
    if latest is None:
        raise FileNotFoundError(
            "EVAL_PATHS is empty and outputs/evaluations/ has no JSONL files. "
            "Run finetune/run_evaluations.py first."
        )
    EVAL_PATHS = [latest]

EVAL_PATHS = [_resolve(p) for p in EVAL_PATHS]
for p in EVAL_PATHS:
    print(p)


In [ ]:
# Load each file into its own DataFrame, tagged with model_type / dataset
# so that downstream plots can group / label per run.
runs: dict[str, dict] = {}
for path in EVAL_PATHS:
    model_type, dataset = _parse_meta(path)
    df, _ = load_evaluation_jsonl(path)
    label = f"{model_type} / {dataset}"
    if label in runs:  # disambiguate duplicates
        i = 2
        while f"{label} #{i}" in runs:
            i += 1
        label = f"{label} #{i}"
    df = df.assign(model_type=model_type, dataset=dataset, run_label=label)
    runs[label] = {"df": df, "model_type": model_type, "dataset": dataset, "path": path}
    print(f"  {label:40s}  n={len(df):5d}   ({path.name})")

combined = (
    pd.concat([r["df"] for r in runs.values()], ignore_index=True)
    if runs else pd.DataFrame()
)
print(f"\nTotal sample rows across runs: {len(combined)}")
combined.head(3)


: 

## Section 2 — Accuracy & position bias

Two checks:
1. **Overall accuracy** per model split. Sanity check that the eval ran and the finetune helped.
2. **Position bias.** A model that always picks position B will have high accuracy on questions where B is correct and chance accuracy elsewhere. Chi-square on predicted-position counts catches the same failure mode independently of correctness.

In [ ]:
def position_diagnostics(df, label):
    if df.empty:
        print(f'{label}: no rows'); return
    n = len(df)
    acc = df['is_correct'].mean()
    print(f'\n=== {label} (n={n}) ===')
    print(f'  overall accuracy: {acc:.2%}')

    pred_counts = df['model_answer_position'].value_counts().reindex([0, 1, 2, 3], fill_value=0)
    expected = n / 4.0
    chi2 = float(((pred_counts - expected) ** 2 / expected).sum())
    verdict = 'uniform' if chi2 < 7.81 else 'biased'
    print(f'  predicted-position counts (pos0–3): {pred_counts.tolist()}')
    print(f'  chi-square: {chi2:.2f}  (crit p=0.05: 7.81 → {verdict})')

    print('  accuracy conditioned on correct position:')
    for p in range(4):
        sub = df[df['correct_answer_position'] == p]
        if not sub.empty:
            print(f'    correct=pos{p}: {sub["is_correct"].mean():.2%}  (n={len(sub)})')

for label, r in runs.items():
    position_diagnostics(r["df"], label)


In [ ]:
# --- Overall accuracy bar chart (one bar per run, labeled model / dataset) ---
labels_list, accs = [], []
for lbl, r in runs.items():
    df = r["df"]
    if df.empty or 'is_correct' not in df.columns:
        continue
    labels_list.append(lbl)
    accs.append(df['is_correct'].mean())

if not labels_list:
    print('No runs with is_correct — skipping overall-accuracy plot.')
else:
    fig, ax = plt.subplots(figsize=(max(5, len(labels_list) * 1.8), 4))
    bars = ax.bar(range(len(labels_list)), accs, color=plt.cm.tab10.colors[:len(labels_list)])
    ax.set_xticks(range(len(labels_list)))
    ax.set_xticklabels(labels_list, rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1)
    ax.axhline(0.25, color='gray', linestyle=':', alpha=0.7, label='chance (25%)')
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{acc:.1%}', ha='center', va='bottom', fontsize=9)
    ax.set_title('Overall accuracy by model / dataset')
    ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()

# --- Per-position conditional accuracy (one panel per run) ---
runs_with_acc = [(lbl, r) for lbl, r in runs.items()
                 if not r["df"].empty
                 and {'is_correct', 'correct_answer_position'}.issubset(r["df"].columns)]

if not runs_with_acc:
    print('No runs with correct_answer_position — skipping per-position plot.')
else:
    n_runs = len(runs_with_acc)
    fig, axes = plt.subplots(1, n_runs, figsize=(4 * n_runs, 4), sharey=True, squeeze=False)
    axes = axes[0]
    pos_labels = ['pos0 (A)', 'pos1 (B)', 'pos2 (C)', 'pos3 (D)']
    for ax, (label, r) in zip(axes, runs_with_acc):
        df = r["df"]
        pos_accs, pos_ns = [], []
        for p in range(4):
            sub = df[df['correct_answer_position'] == p]
            pos_accs.append(sub['is_correct'].mean() if len(sub) else float('nan'))
            pos_ns.append(len(sub))
        bars = ax.bar(pos_labels, pos_accs, color=plt.cm.tab10.colors[:4])
        ax.axhline(df['is_correct'].mean(), color='black', linestyle='--', linewidth=1, label='overall avg')
        ax.set_ylim(0, 1)
        ax.set_ylabel('Accuracy')
        ax.set_title(f'{label}\nn={len(df)}')
        for bar, acc, n in zip(bars, pos_accs, pos_ns):
            if not np.isnan(acc):
                ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.01,
                        f'{acc:.1%}\n(n={n})', ha='center', va='bottom', fontsize=8)
        ax.legend(fontsize=8)
    fig.suptitle('Accuracy conditioned on correct answer position')
    fig.tight_layout(); plt.show()

In [ ]:
# Predicted-position counts, one panel per run.
# sharey=False so each dataset's distribution is readable on its own scale.
n_runs = max(len(runs), 1)
fig, axes = plt.subplots(1, n_runs, figsize=(4 * n_runs, 4), sharey=False, squeeze=False)
axes = axes[0]
for ax, (label, r) in zip(axes, runs.items()):
    df = r["df"]
    if df.empty:
        ax.set_title(f'{label}\n(no data)'); continue
    counts = df['model_answer_position'].value_counts().reindex([0, 1, 2, 3], fill_value=0)
    ax.bar(['A', 'B', 'C', 'D'], counts.values)
    ax.axhline(len(df) / 4.0, color='gray', linestyle=':', label='uniform')
    ax.set_title(f'{label}\nn={len(df)}')
    ax.set_ylabel('count')
    ax.legend(fontsize=8)
fig.suptitle('Predicted answer position counts')
fig.tight_layout(); plt.show()

## Section 3 — Calibration & metacognition

Calibration of the model's *internal* probability (`top_prob`) and its *verbalized* probability (`expected_confidence`):

- Reliability diagram on `top_prob` vs. `is_correct`, plus ECE / Brier.
- Spearman(entropy, stated_confidence) — the standard metacognition signal: high (negative) correlation means stated confidence tracks internal uncertainty.
- Three overlaid plots that unpack the relationships between **entropy**, **stated confidence**, and **correctness** (scatter + reliability + AUROC).

In [ ]:
def calibration_panel(df, label):
    if df.empty or 'top_prob' not in df.columns:
        print(f'{label}: missing top_prob'); return None
    valid = df.dropna(subset=['top_prob', 'is_correct'])
    conf = valid['top_prob'].to_numpy()
    correct = valid['is_correct'].astype(float).to_numpy()
    tab = calibration_table(conf, correct)
    ece = expected_calibration_error(conf, correct)
    brier = brier_score(conf, correct)
    print(f'{label}: ECE={ece:.4f}  Brier={brier:.4f}  (n={len(valid)})')
    return tab

tabs = {label: calibration_panel(r["df"], label) for label, r in runs.items()}

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], color='gray', linestyle=':', label='perfect calibration')
for label, tab in tabs.items():
    if tab is None or tab.empty:
        continue
    ax.plot(tab['mean_conf'], tab['accuracy'], 'o-', label=label)
ax.set_xlabel('mean predicted probability (top_prob)')
ax.set_ylabel('empirical accuracy')
ax.set_title('Reliability diagram')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


In [ ]:
# Metacognition: Spearman(entropy, stated_confidence). Negative = the model's
# stated confidence tracks its internal uncertainty (more entropy → less
# confidence). Requires the eval to have been run with stated-confidence on.
for label, r in runs.items():
    df = r["df"]
    if df.empty or 'expected_confidence' not in df.columns or 'entropy' not in df.columns:
        print(f'{label}: skipping (no stated-confidence column)'); continue
    valid = df.dropna(subset=['entropy', 'expected_confidence'])
    if valid.empty:
        print(f'{label}: no valid rows'); continue
    rho, p = spearmanr(valid['entropy'], valid['expected_confidence'])
    print(f'{label}: Spearman(entropy, stated_conf) = {rho:+.3f}  (p={p:.2g}, n={len(valid)})')


### Relationships between entropy, stated confidence, and correctness

Spearman alone doesn't show the *shape* of the relationship. Three diagnostics:

1. **Entropy vs. stated confidence** — scatter (one panel per run). A model with introspective access produces a clean monotone band running from high stated confidence at low entropy to low stated confidence at high entropy.
2. **Stated confidence vs. correctness** — reliability diagram (overlaid across runs). Whether the verbalized probability is calibrated against actual accuracy.
3. **Entropy vs. correctness** — accuracy binned by entropy (overlaid across runs). The internal-uncertainty ceiling: stated-confidence calibration can't be a better correctness predictor than this.

In [ ]:
# --- 1. Entropy vs. stated confidence (scatter per run) ---
runs_meta = [(lbl, r) for lbl, r in runs.items()
             if {'entropy', 'expected_confidence'}.issubset(r["df"].columns)
             and r["df"][['entropy', 'expected_confidence']].dropna().shape[0] > 0]

if runs_meta:
    n = len(runs_meta)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharex=True, sharey=True, squeeze=False)
    axes = axes[0]
    for ax, (lbl, r) in zip(axes, runs_meta):
        v = r["df"][['entropy', 'expected_confidence', 'is_correct']].dropna()
        colors = v['is_correct'].map({True: 'tab:green', False: 'tab:red'})
        ax.scatter(v['entropy'], v['expected_confidence'], s=6, alpha=0.3, c=colors)
        ax.set_xlabel('entropy (bits)')
        ax.set_ylabel('stated confidence')
        ax.set_title(f'{lbl}\nn={len(v)}')
    fig.suptitle('Entropy vs. stated confidence  (green=correct, red=wrong)')
    fig.tight_layout(); plt.show()
else:
    print('No runs have both entropy and expected_confidence.')


In [ ]:
# --- 2. Stated confidence vs. correctness (reliability diagram, overlaid) ---
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], color='gray', linestyle=':', label='perfect calibration')
for lbl, r in runs.items():
    df = r["df"]
    if 'expected_confidence' not in df.columns:
        continue
    valid = df.dropna(subset=['expected_confidence', 'is_correct'])
    if valid.empty:
        continue
    conf = valid['expected_confidence'].to_numpy()
    correct = valid['is_correct'].astype(float).to_numpy()
    tab = calibration_table(conf, correct)
    ece = expected_calibration_error(conf, correct)
    brier = brier_score(conf, correct)
    if not tab.empty:
        ax.plot(tab['mean_conf'], tab['accuracy'], 'o-',
                label=f'{lbl}  ECE={ece:.3f} Brier={brier:.3f}')
ax.set_xlabel('mean stated confidence')
ax.set_ylabel('empirical accuracy')
ax.set_title('Stated confidence vs. correctness')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


In [ ]:
# --- 3. Entropy vs. correctness (accuracy in entropy bins, overlaid) ---
# Bins span [0, log2(4)=2]. Accuracy in each bin = fraction of correct answers
# among questions whose MC-entropy falls in that bin. AUROC of -entropy vs.
# is_correct is the headline summary stat (1.0 = entropy perfectly orders
# correct vs. wrong, 0.5 = no information).
try:
    from sklearn.metrics import roc_auc_score
    _have_sklearn = True
except ImportError:
    _have_sklearn = False

ent_bin_edges = np.linspace(0.0, 2.0, 11)
ent_bin_mids = 0.5 * (ent_bin_edges[:-1] + ent_bin_edges[1:])

fig, ax = plt.subplots(figsize=(7, 5))
for lbl, r in runs.items():
    df = r["df"]
    if 'entropy' not in df.columns:
        continue
    valid = df.dropna(subset=['entropy', 'is_correct'])
    if valid.empty:
        continue
    ent = valid['entropy'].to_numpy()
    correct = valid['is_correct'].astype(float).to_numpy()
    bin_ix = np.clip(np.digitize(ent, ent_bin_edges) - 1, 0, len(ent_bin_mids) - 1)
    accs, mids, ns = [], [], []
    for b in range(len(ent_bin_mids)):
        mask = bin_ix == b
        if mask.sum() == 0:
            continue
        accs.append(correct[mask].mean())
        mids.append(ent_bin_mids[b])
        ns.append(int(mask.sum()))
    auc_str = ''
    if _have_sklearn and len(np.unique(correct)) == 2:
        auroc = roc_auc_score(correct, -ent)
        auc_str = f'  AUROC(-ent)={auroc:.3f}'
    ax.plot(mids, accs, 'o-', label=f'{lbl}{auc_str}')
ax.set_xlabel('entropy (bits)')
ax.set_ylabel('empirical accuracy')
ax.set_title('Entropy vs. correctness  (lower entropy → higher accuracy if model is well-calibrated internally)')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


### Calibration + metacognition summary  (one panel per metric, x = model, bar per dataset)

Replicates the `Section 1.9` summary plot from the legacy notebook. Each metric is computed per `(model_type, dataset)` run and rendered as a grouped bar chart.

- **Calibration** (does stated confidence match accuracy?) — accuracy, mean stated confidence, ECE, Brier, mean(conf) − mean(acc).
- **Resolution** (does stated confidence rank correct vs. wrong?) — `AUROC(stated, correct)`, `ρ(stated, correct)`. Invariant to monotonic transforms of the scale.
- **Internal-uncertainty ceiling** — `AUROC(−H, correct)`. Stated-confidence AUROC can't exceed this; closing the gap is what introspective access has to do.
- **Metacognition correlations** — `ρ(stated, −entropy)`, `ρ(stated, prob_gap)`, `ρ(stated, top_prob)`.

In [ ]:
from scipy.stats import spearmanr, pearsonr
try:
    from sklearn.metrics import roc_auc_score
    _have_sklearn = True
except ImportError:
    _have_sklearn = False


def _safe_array(s):
    return np.asarray(s, dtype=float)


def _ece(conf, correct, n_bins=10):
    conf, correct = _safe_array(conf), _safe_array(correct)
    m = ~np.isnan(conf) & ~np.isnan(correct)
    conf, correct = conf[m], correct[m]
    if len(conf) == 0:
        return float('nan')
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, edges) - 1, 0, n_bins - 1)
    n = len(conf)
    ece = 0.0
    for b in range(n_bins):
        sel = idx == b
        if sel.sum() == 0:
            continue
        ece += (sel.sum() / n) * abs(conf[sel].mean() - correct[sel].mean())
    return float(ece)


def _brier(conf, correct):
    conf, correct = _safe_array(conf), _safe_array(correct)
    m = ~np.isnan(conf) & ~np.isnan(correct)
    if m.sum() == 0:
        return float('nan')
    return float(np.mean((conf[m] - correct[m]) ** 2))


def _auroc(score, correct):
    if not _have_sklearn:
        return float('nan')
    score, correct = _safe_array(score), _safe_array(correct)
    m = ~np.isnan(score) & ~np.isnan(correct)
    if m.sum() < 10 or len(np.unique(correct[m])) < 2:
        return float('nan')
    return float(roc_auc_score(correct[m].astype(int), score[m]))


def _spear(a, b):
    a, b = _safe_array(a), _safe_array(b)
    m = ~np.isnan(a) & ~np.isnan(b)
    if m.sum() < 10:
        return float('nan')
    res = spearmanr(a[m], b[m])
    return float(getattr(res, 'correlation', getattr(res, 'statistic', np.nan)))


def _pears(a, b):
    a, b = _safe_array(a), _safe_array(b)
    m = ~np.isnan(a) & ~np.isnan(b)
    if m.sum() < 10:
        return float('nan')
    return float(pearsonr(a[m], b[m])[0])


# Build per-(model, dataset) metrics row.
META_METRICS = []
for label, r in runs.items():
    df = r["df"]
    if df.empty:
        continue
    conf = df['expected_confidence'].to_numpy() if 'expected_confidence' in df.columns else None
    correct = df['is_correct'].astype(float).to_numpy() if 'is_correct' in df.columns else None
    entropy = df['entropy'].to_numpy() if 'entropy' in df.columns else None
    top_prob = df['top_prob'].to_numpy() if 'top_prob' in df.columns else None
    prob_gap = df['prob_gap'].to_numpy() if 'prob_gap' in df.columns else None

    if conf is None or correct is None:
        continue
    valid = ~np.isnan(conf) & ~np.isnan(correct)
    c, y = conf[valid], correct[valid]

    META_METRICS.append({
        'model':   r["model_type"],
        'dataset': r["dataset"],
        'n':       int(valid.sum()),
        'accuracy':           float(y.mean()) if len(y) else float('nan'),
        'mean_stated_conf':   float(c.mean()) if len(c) else float('nan'),
        'ECE':                _ece(c, y),
        'Brier':              _brier(c, y),
        'overconfidence':     float(c.mean() - y.mean()) if len(c) and len(y) else float('nan'),
        'AUROC(stated)':      _auroc(conf, correct),
        'AUROC(-H)':          _auroc(-entropy, correct) if entropy is not None else float('nan'),
        'rho(stated,correct)':   _spear(conf, correct),
        'rho(stated,-entropy)':  _spear(conf, -entropy) if entropy is not None else float('nan'),
        'pearson(stated,-entropy)': _pears(conf, -entropy) if entropy is not None else float('nan'),
        'rho(stated,prob_gap)':  _spear(conf, prob_gap) if prob_gap is not None else float('nan'),
        'rho(stated,top_prob)':  _spear(conf, top_prob) if top_prob is not None else float('nan'),
    })

META_DF = pd.DataFrame(META_METRICS)
print('=== Calibration + metacognition summary ===')
print(META_DF.round(3).to_string(index=False))


# Grouped-bar plot: panel per metric, x = model_type, one bar per dataset.
metrics_to_plot = [
    ('accuracy',                'Accuracy  (↑ better)'),
    ('mean_stated_conf',        'Mean stated confidence'),
    ('ECE',                     'ECE  (↓ better)'),
    ('Brier',                   'Brier score  (↓ better)'),
    ('overconfidence',          'Mean conf − mean acc  (0 = calibrated)'),
    ('AUROC(stated)',           'AUROC(stated, correct) — resolution  (↑ better)'),
    ('AUROC(-H)',               'AUROC(−H, correct) — entropy ceiling  (↑ better)'),
    ('rho(stated,correct)',     'ρ(stated, correct) — resolution  (↑ better)'),
    ('rho(stated,-entropy)',    'ρ(stated, −entropy)  (↑ better)'),
    ('pearson(stated,-entropy)','Pearson r(stated, −entropy)  (↑ better)'),
    ('rho(stated,prob_gap)',    'ρ(stated, prob_gap)  (↑ better)'),
    ('rho(stated,top_prob)',    'ρ(stated, top_prob)  (↑ better)'),
]

if META_DF.empty:
    print('No (model, dataset) runs with stated_confidence — nothing to plot.')
else:
    models_present   = list(dict.fromkeys(META_DF['model'].tolist()))
    datasets_present = list(dict.fromkeys(META_DF['dataset'].tolist()))
    n_ds = max(1, len(datasets_present))
    width = 0.8 / n_ds
    x = np.arange(len(models_present))

    n_metrics = len(metrics_to_plot)
    n_cols = 4
    n_rows = (n_metrics + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.3 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for i, (key, title) in enumerate(metrics_to_plot):
        ax = axes[i]
        for j, ds in enumerate(datasets_present):
            vals = []
            for m in models_present:
                sub = META_DF[(META_DF['model'] == m) & (META_DF['dataset'] == ds)]
                vals.append(sub[key].iloc[0] if len(sub) else float('nan'))
            offset = (j - (n_ds - 1) / 2) * width
            ax.bar(x + offset, vals, width=width * 0.95, label=ds)
        ax.set_xticks(x); ax.set_xticklabels(models_present, fontsize=8)
        ax.set_title(title, fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')
        if key.startswith('rho') or key.startswith('pearson') or key == 'overconfidence':
            ax.axhline(0, color='gray', linestyle=':', alpha=0.6)
        if key.startswith('AUROC'):
            ax.axhline(0.5, color='gray', linestyle=':', alpha=0.6)
        if i == 0:
            ax.legend(fontsize=8, title='dataset', loc='best')
    for k in range(n_metrics, len(axes)):
        axes[k].axis('off')
    plt.suptitle('Calibration and metacognition — all models, all datasets',
                 y=1.02, fontsize=12)
    plt.tight_layout(); plt.show()


## Section 4 — Confidence answer distribution

Bar charts of the model's predicted confidence-bin index, on both the self-confidence pass (`predicted_confidence_bin_index`) and the other-confidence pass (`predicted_other_confidence_bin_index`). One panel per run, labeled by `model_type / dataset`.

Bins are 0-indexed in the JSONL (1..N on the x-axis below); the bin count comes from `confidence_format` in `finetune/loss.py` — currently `numeric_1_10` (10 bins of 10% each).

A reasonable model spreads mass across bins; pathological behavior shows up as a spike on one bin (e.g., always picking the highest).

In [ ]:
# Confidence-bin distribution. Bins are 0-indexed in the JSONL; we display
# them as 1..N (e.g. 1..10 for the numeric_1_10 scheme). Bin count is read
# from the data so this works for any confidence_format in finetune/loss.py.
# sharey=False so each dataset's distribution is readable on its own scale.
def plot_conf_dist(field, title):
    runs_with = [(lbl, r) for lbl, r in runs.items()
                 if field in r["df"].columns and r["df"][field].notna().any()]
    if not runs_with:
        print(f'No runs have {field}'); return
    n_bins = max(int(r["df"][field].dropna().max()) for _, r in runs_with) + 1
    n_bins = max(n_bins, 1)
    labels = [str(i + 1) for i in range(n_bins)]
    n = len(runs_with)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=False, squeeze=False)
    axes = axes[0]
    for ax, (lbl, r) in zip(axes, runs_with):
        s = r["df"][field].dropna().astype(int)
        counts = s.value_counts().reindex(range(n_bins), fill_value=0)
        ax.bar(range(n_bins), counts.values)
        ax.set_xticks(range(n_bins))
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_xlabel('confidence bin (1 = lowest)')
        ax.set_title(f'{lbl}\nn={len(s)}')
        ax.set_ylabel('count')
    fig.suptitle(title)
    fig.tight_layout(); plt.show()


plot_conf_dist('predicted_confidence_bin_index',
               'Self-confidence answer distribution')
plot_conf_dist('predicted_other_confidence_bin_index',
               'Other-confidence answer distribution')

## Section 5 — Entropy distribution

Histogram of per-question MC-answer entropy (bits over the four-way A/B/C/D distribution; max = log2(4) = 2). Overlaid across runs so you can see whether the model is getting peakier (low entropy = confident, often after a calibration finetune) or flatter (high entropy = uncertain).

In [ ]:
# Overlay entropy histograms across all runs. Density-normalized so runs with
# different N stay visually comparable.
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0.0, 2.0, 41)
any_plotted = False
for label, r in runs.items():
    df = r["df"]
    if 'entropy' not in df.columns or df['entropy'].isna().all():
        continue
    ax.hist(df['entropy'].dropna(), bins=bins, alpha=0.4, density=True, label=label)
    any_plotted = True
if any_plotted:
    ax.set_xlabel('entropy (bits)')
    ax.set_ylabel('density')
    ax.set_title('MC answer-distribution entropy')
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('No runs have an entropy column.')

# Per-run summary stats for context.
for label, r in runs.items():
    df = r["df"]
    if 'entropy' in df.columns and df['entropy'].notna().any():
        e = df['entropy'].dropna()
        print(f'{label}: entropy  mean={e.mean():.3f}  median={e.median():.3f}  '
              f'p10={e.quantile(0.10):.3f}  p90={e.quantile(0.90):.3f}  (n={len(e)})')


## Section 6 — Delegation game (placeholder)

The delegate-game pipeline was removed in commit `8f1441d`. Once it's rebuilt for the new finetune, the analysis would go here — typical plot is delegate-rate vs. confidence bin. The legacy implementation lives in [`legacy_code/analyze_activations_pre_split.ipynb`](legacy_code/analyze_activations_pre_split.ipynb) under "Section 1.10".

## Section 7 — Explicit Confidence Task (ECT) analysis

Reads ECT result files (one JSON per run condition) from `outputs/ECT/`. Each row in a file has the model's MC probabilities, its stated confidence on the S–Z scale, the predicted answer letter, and `is_correct`. We compute:

- Stated-confidence calibration: ECE, Brier, AUROC against `is_correct`.
- **MC-entropy AUROC ceiling.** How well does the model's MC entropy alone predict correctness? Stated-confidence AUROC can't exceed this. Closing the gap = introspective access; raising the ceiling = better underlying calibration.
- Spearman / Pearson correlation between stated confidence and inverted MC entropy.

Originally lived in `analyze_ECT.ipynb` (now in `legacy_code/`).

In [ ]:
from analysis_helpers import load_ect_results
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score

ECT_DIR = REPO_ROOT / 'outputs' / 'ECT'
ect_files = sorted(ECT_DIR.glob('*.json')) if ECT_DIR.exists() else []
print(f'Found {len(ect_files)} ECT files in {ECT_DIR}')

if not ect_files:
    print('No ECT result files yet. Run the ECT pipeline first.')
else:
    ect = load_ect_results(ect_files)
    rows = []
    for name, blob in ect.items():
        data = blob.get('data', [])
        if not data:
            continue
        is_correct = np.array([int(d['is_correct']) for d in data])
        stated = np.array([float(d['stated_confidence_value']) for d in data])
        mc_entropy = np.array([
            -np.sum(np.array(d['mc_probs']) * np.log2(np.maximum(np.array(d['mc_probs']), 1e-12)))
            for d in data
        ])
        # Calibration of stated confidence (assumes stated is in [0, 1] already).
        ece = expected_calibration_error(stated, is_correct.astype(float))
        brier = brier_score(stated, is_correct.astype(float))
        auroc_stated = (roc_auc_score(is_correct, stated)
                        if len(np.unique(is_correct)) == 2 else float('nan'))
        # Ceiling: how well does internal entropy predict correctness?
        auroc_entropy = (roc_auc_score(is_correct, -mc_entropy)
                         if len(np.unique(is_correct)) == 2 else float('nan'))
        rho, _ = pearsonr(stated, -mc_entropy)
        rows.append({
            'run': name,
            'n': len(is_correct),
            'accuracy': is_correct.mean(),
            'mean_stated_conf': stated.mean(),
            'ece_stated': ece,
            'brier_stated': brier,
            'auroc_stated': auroc_stated,
            'auroc_entropy_ceiling': auroc_entropy,
            'pearson(stated, -entropy)': rho,
        })
    pd.DataFrame(rows).set_index('run')

---

**See also:** [`analyze_activations.ipynb`](analyze_activations.ipynb) for probe / direction analysis, [`analyze_interventions.ipynb`](analyze_interventions.ipynb) for ablation & steering analysis, [`legacy_code/`](legacy_code/) for the older implementations these notebooks superseded.